# ITAMAE: cosmological scales, a Milky Way halo and a bound orbit

## Goal
Use the shared numerical components with physical inputs: a Planck18 background,
a normalized linear power spectrum, an NFW halo of $M_{200}=10^{12}M_\odot$, and a
bound orbit between 30 and 150 kpc. The notebook produces expansion/growth curves,
$\sigma(M)$, a rotation curve and the time spent in radial shells.

These examples demonstrate mechanisms in physical units. The chosen NFW profile
and one orbit are explicit idealizations, not a new SASHIMI spatial prescription
or a cosmological distribution of satellite orbits.

## Setup
From the migration checkout:
```sh
uv sync --extra demo
uv run --no-sync python -m ipykernel install --user --name itamae-demo --display-name "ITAMAE demo"
```
Select that kernel and **Restart Kernel and Run All**. The optional Colossus
package supplies an analytic Eisenstein–Hu spectrum; no external data is downloaded.
The basic execution/catalog API examples are retained in
[archive/api_primitives.ipynb](archive/api_primitives.ipynb).

In [ ]:
%matplotlib inline
import sys
import json
from pathlib import Path
from importlib.metadata import version
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from itamae.provenance import source_revision
import itamae

plt.rcParams.update({"figure.figsize": (7.2, 4.5), "font.size": 11,
                     "axes.grid": True, "grid.alpha": 0.18,
                     "figure.constrained_layout.use": True})
COLORS = ["#0072B2", "#D55E00", "#009E73", "#555555"]
STYLES = ["-", "--", "-.", ":"]
output_dir = Path("outputs/usage_walkthrough")
output_dir.mkdir(parents=True, exist_ok=True)


def table(headers, rows):
    display(Markdown("| " + " | ".join(headers) + " |\n| " +
                     " | ".join(["---"] * len(headers)) + " |\n" +
                     "\n".join("| " + " | ".join(map(str, row)) + " |" for row in rows)))

provenance = {"version": version("sashimi-itamae"),
              "source_revision": source_revision("itamae", module_file=itamae.__file__),
              "numpy": version("numpy"), "scipy": version("scipy"), "colossus": version("colossus")}
table(["Environment", "Value"], provenance.items())

## 1. Set a physical cosmology

Use Colossus's `planck18` parameter set for the input spectrum, and the same
$\Omega_{m,0}$ and $h$ in the native flat matter-plus-$\Lambda$ background.
ITAMAE's native growth uses the normalized Carroll–Press–Turner approximation;
radiation is neglected here. The example is restricted to $0\le z\le7$.

In [ ]:
from colossus.cosmology import cosmology as colossus_cosmology
from itamae.cosmology import NativeFlatLCDM
external = colossus_cosmology.Cosmology(name="itamae-demo-planck18", persistence="",
                                       **dict(colossus_cosmology.cosmologies["planck18"]))
cosmology = NativeFlatLCDM(omega_m0=external.Om0, h=external.h)
z = np.linspace(0, 7, 150)
np.testing.assert_allclose(cosmology.growth_factor(0), 1)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for ax, values, label in [(axes[0], cosmology.H(z), "H(z) [km/s/Mpc]"),
                           (axes[1], cosmology.growth_factor(z), "D(z), normalized to D(0)=1"),
                           (axes[2], cosmology.cosmic_time(z), "Cosmic age [Gyr]")]:
    ax.plot(z, values)
    ax.set(xlabel="Redshift", ylabel=label)
fig.suptitle("Planck18 matter-plus-Lambda background")
plt.show()
table(["Parameter", "Value"], [("Omega_m,0", external.Om0), ("h", external.h),
                                ("sigma_8", external.sigma8), ("n_s", external.ns),
                                ("Age at z=0 [Gyr]", f"{float(cosmology.cosmic_time(0)):.4f}")])

## 2. Integrate a normalized power spectrum

[Colossus's documented `matterPowerSpectrum`](https://bdiemer.bitbucket.io/colossus/cosmology_cosmology.html#cosmology.cosmology.Cosmology.matterPowerSpectrum)
returns the analytic Eisenstein–Hu model in $(\mathrm{Mpc}/h)^3$ with $k$ in $h$/Mpc.
Convert both to physical Mpc units before passing the table to ITAMAE.
The top-hat variance uses the present mean matter density,
$R=(3M/4\pi\rho_{m,0})^{1/3}$, and $S(M,z)=D(z)^2S(M,0)$.
The spectrum is a fitting formula, not a CAMB/CLASS transfer calculation.

In [ ]:
from itamae.power import TabulatedPowerSpectrum, SphericalTopHatWindow
from itamae.variance import IntegratedVarianceModel
k_h = np.geomspace(1e-5, 1e4, 8193)
power_h = external.matterPowerSpectrum(k_h, z=0, model="eisenstein98")
k = k_h * external.h
power_mpc = power_h / external.h**3
power = TabulatedPowerSpectrum(k, power_mpc, identifier="Colossus:planck18:eisenstein98;units=Mpc,Msun")
variance = IntegratedVarianceModel(power, SphericalTopHatWindow(), rho_mean=float(cosmology.rho_m(0)),
                                   k_min=k[0], k_max=k[-1], n_k=8193,
                                   growth_function=cosmology.growth_factor,
                                   growth_identifier=cosmology.identifier+":CPT-normalized")
masses = np.geomspace(1e6, 1e15, 100)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.3))
axes[0].loglog(k, power_mpc)
axes[0].set(xlabel=r"k [Mpc$^{-1}$]", ylabel=r"P(k) [Mpc$^3$]", title="Normalized linear power at z=0")
for redshift, color, style in zip([0, 1, 3], COLORS, STYLES):
    sigma = variance.sigma(masses, redshift)
    axes[1].loglog(masses, sigma, style, color=color, label=f"z = {redshift}")
axes[1].set(xlabel=r"Mass [$M_\odot$]", ylabel=r"$\sigma(M,z)$", title="Top-hat smoothed fluctuations")
axes[1].legend()
plt.show()
radius8 = 8/external.h
mass8 = 4*np.pi/3*cosmology.rho_m(0)*radius8**3
sigma8 = float(variance.sigma(mass8))
np.testing.assert_allclose(sigma8, external.sigma8, rtol=2e-3)
np.testing.assert_allclose(variance.variance(masses, 1), variance.variance(masses, 0)*cosmology.growth_factor(1)**2)
table(["Normalization check", "Value"], [("Integrated sigma(8/h Mpc)", f"{sigma8:.6f}"),
                                         ("Input sigma_8", external.sigma8)])

## 3. Construct a Milky Way scale NFW halo

Choose $M_{200}=10^{12}M_\odot$ and concentration $c_{200}=10$ at $z=0$.
Derive $r_{200}$ from $200\rho_{crit}$, then normalize $\rho_s$ so the enclosed
mass at $r_{200}$ equals the input mass. The circular speed is
$V_c(r)=\sqrt{GM(<r)/r}$; the baryonic disk and bulge are absent.

In [ ]:
from itamae.halo import NFWProfile, nfw_mass_function
G = 4.30091e-9  # Mpc (km/s)^2 / Msun; the ITAMAE NFW convention
host_mass, concentration = 1e12, 10.
r200 = (3*host_mass/(4*np.pi*200*cosmology.rho_crit(0)))**(1/3)
rs = r200/concentration
rhos = host_mass/(4*np.pi*rs**3*nfw_mass_function(concentration))
profile = NFWProfile(r_s=rs, rho_s=rhos)
np.testing.assert_allclose(profile.enclosed_mass(r200), host_mass, rtol=1e-13)
radius = np.geomspace(.001, r200, 300)  # Mpc
speed = np.sqrt(G*profile.enclosed_mass(radius)/radius)
rmax = 2.1625815870646098*rs
vmax = np.sqrt(G*profile.enclosed_mass(rmax)/rmax)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.3))
axes[0].loglog(radius*1000, profile.density(radius)/1e18)
axes[0].set(xlabel="Radius [kpc]", ylabel=r"Density [$M_\odot$ pc$^{-3}$]", title="NFW density profile")
axes[1].semilogx(radius*1000, speed)
axes[1].plot(rmax*1000, vmax, "o", color=COLORS[1], label=f"Vmax = {vmax:.1f} km/s")
axes[1].axvline(rmax*1000, ls=":", color=COLORS[1])
axes[1].set(xlabel="Radius [kpc]", ylabel="Circular speed [km/s]", title="Dark-matter-only rotation curve")
axes[1].legend()
plt.show()
table(["Halo scale", "Value"], [("r200 [kpc]", f"{1000*r200:.3f}"),
                                 ("rs [kpc]", f"{1000*rs:.3f}"),
                                 ("rmax [kpc]", f"{1000*rmax:.3f}"), ("Vmax [km/s]", f"{vmax:.3f}")])

## 4. Follow one bound orbit in that potential

Set pericenter 30 kpc and apocenter 150 kpc in the static NFW potential.
Solve the two turning-point equations for specific energy and angular momentum,
then recover the turning points and radial period with ITAMAE. The shell weights
are the fraction of orbital time in each shell; division by shell width gives a
probability density in kpc$^{-1}$. This is one test-particle orbit with fixed
invariants, without stripping, dynamical friction or host evolution.

In [ ]:
from itamae.spatial import turning_points, radial_period, orbit_radial_measure
rp_input, ra_input = .030, .150  # Mpc
angular_momentum2 = 2*(profile.potential(ra_input)-profile.potential(rp_input))/(1/rp_input**2-1/ra_input**2)
angular_momentum = np.sqrt(angular_momentum2)
energy = profile.potential(rp_input)+angular_momentum2/(2*rp_input**2)
rp, ra = turning_points(profile.potential, energy, angular_momentum, .001, 1.)
np.testing.assert_allclose([rp, ra], [rp_input, ra_input], rtol=1e-9)
mpc_per_kms_to_gyr = 3.0856775814913673e19 / (365.25*86400*1e9)
period_gyr = radial_period(profile.potential, energy, angular_momentum, rp, ra)*mpc_per_kms_to_gyr
edges = np.linspace(rp, ra, 41)
measure = orbit_radial_measure(edges, profile.potential, energy, angular_momentum, rp, ra)
np.testing.assert_allclose(measure.weight.sum(), 1, rtol=1e-12)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.3))
r = np.geomspace(.01, .25, 400)
axes[0].plot(r*1000, profile.potential(r)+angular_momentum2/(2*r**2), label="Effective potential")
axes[0].axhline(energy, color=COLORS[1], ls="--", label="Orbital energy")
axes[0].plot([rp*1000, ra*1000], [energy, energy], "o", color=COLORS[1])
axes[0].set(xlabel="Radius [kpc]", ylabel=r"Specific energy [(km/s)$^2$]", title="Turning points")
minimum = float(np.min(profile.potential(r)+angular_momentum2/(2*r**2)))
axes[0].set_ylim(minimum-.1*abs(minimum), energy+.5*(energy-minimum))
axes[0].legend(fontsize=9)
axes[1].stairs(measure.weight / np.diff(edges*1000), edges*1000)
axes[1].set(xlabel="Radius [kpc]", ylabel=r"Orbital probability [kpc$^{-1}$]", title="Time spent in radial shells")
plt.show()
table(["Orbit", "Value"], [("Pericenter [kpc]", f"{rp*1000:.4f}"),
                            ("Apocenter [kpc]", f"{ra*1000:.4f}"),
                            ("Radial period [Gyr]", f"{period_gyr:.4f}")])

## Checks

Verify the period implementation against a physical Kepler orbit, whose exact
period is known. Also halve the power-integration sampling to report its finite-grid
change at the displayed masses. These numerical checks are separate from the
accuracy of the analytic spectrum and background approximations.

In [ ]:
from dataclasses import replace
point_mass = 1e12
semi_major, eccentricity = .100, .5
kepler_potential = lambda r: -G*point_mass/np.asarray(r)
kepler_energy = -G*point_mass/(2*semi_major)
kepler_angular_momentum = np.sqrt(G*point_mass*semi_major*(1-eccentricity**2))
kepler_period = radial_period(kepler_potential, kepler_energy, kepler_angular_momentum,
                              semi_major*(1-eccentricity), semi_major*(1+eccentricity))
exact_period = 2*np.pi*np.sqrt(semi_major**3/(G*point_mass))
np.testing.assert_allclose(kepler_period, exact_period, rtol=1e-8)
coarser_sigma = replace(variance, n_k=4097).sigma(masses)
grid_change = float(np.max(np.abs(variance.sigma(masses)/coarser_sigma-1)))
table(["Check", "Result"], [("Kepler period relative error", f"{abs(kepler_period/exact_period-1):.3g}"),
                            ("Largest sigma change, n_k 4097 to 8193", f"{grid_change:.3%}")])
summary = {"provenance": provenance, "halo_mass_Msun": host_mass, "concentration": concentration,
           "r200_kpc": float(1000*r200), "rmax_kpc": float(1000*rmax), "vmax_kms": float(vmax),
           "sigma8": sigma8, "radial_period_Gyr": float(period_gyr), "variance_grid_change": grid_change}
(output_dir/"physical-example-summary.json").write_text(json.dumps(summary, indent=2)+"\n")
print("Physical checks passed; summary saved in", output_dir)

## Next steps

The units, normalization, mass scale and orbital invariants are explicit and the
checks above are independently computable. The SASHIMI-C/SI/W/F walkthroughs use
ITAMAE to predict subhalo mass functions and weighted $V_{max}$–$r_{max}$ distributions
with their model-specific accretion, stripping and survival prescriptions.

For the analytic spectrum's model and conventions, see the
[Colossus power-spectrum documentation](https://bdiemer.bitbucket.io/colossus/cosmology_cosmology.html).
The native cosmology and generic orbit kernels intentionally keep the stated
approximations; this notebook does not assign a new physical default to any variant.